# Predicción de la Producción Agrícola Municipal (EVA 2019–2025)
### Análisis Exploratorio de Datos (EDA) — Primer Corte

**Inteligencia Artificial I — Grupo C2 — Periodo 2026-2**
**Profesor:** Santiago Gómez

**Integrantes del grupo:**
1. _Nombre completo 1_
2. _Nombre completo 2_
3. _Nombre completo 3_
4. _Nombre completo 4_

**Fecha de entrega:** viernes 18 de septiembre de 2026

**Enlace público a este Google Colab:** _(pegar aquí el enlace de "Compartir" antes de presentar)_

---


## 1. Problema y Datos

### 1.1 Origen de los datos
El dataset corresponde a las **Evaluaciones Agropecuarias Municipales (EVA) 2019–2025**, publicadas
por el **Ministerio de Agricultura y Desarrollo Rural de Colombia** (a través de la UPRA / DANE, en
el portal de datos abiertos `datos.gov.co`). La EVA es el registro oficial y censal que usa el
gobierno colombiano para caracterizar, año a año y semestre a semestre, el comportamiento del
sector agrícola en cada uno de los municipios del país.

Cada fila del dataset representa la evaluación de **un cultivo, en un municipio, en un periodo
(año/semestre) específico**, e incluye:

| Columna | Descripción |
|---|---|
| `Departamento`, `Municipio` (+ códigos DANE) | Ubicación geográfica |
| `Grupo cultivo`, `Subgrupo`, `Cultivo`, `Desagregación cultivo` | Clasificación agrícola |
| `Año`, `Periodo` | Año (2019-2025) y periodo (anual, "A" primer semestre, "B" segundo semestre) |
| `Área sembrada` (ha) | Hectáreas sembradas |
| `Área cosechada` (ha) | Hectáreas efectivamente cosechadas |
| **`Producción` (t)** | **Variable objetivo** — toneladas producidas |
| `Rendimiento` (t/ha) | Producción por hectárea cosechada |
| `Ciclo del cultivo` | Transitorio / Permanente |
| `Estado físico del cultivo` | Presentación del producto (fresco, grano seco, etc.) |

### 1.2 Importancia del problema
La agricultura es un sector estratégico para la seguridad alimentaria y la economía regional de
Colombia. Poder **anticipar la producción agrícola** de un cultivo en un municipio permite:

- Apoyar decisiones de política pública (subsidios, créditos, alertas tempranas de escasez).
- Ayudar a productores y comercializadores a planear siembra, almacenamiento y logística.
- Detectar municipios/cultivos con caídas de productividad que requieren intervención.

### 1.3 Desafíos del dataset
- **Formato numérico colombiano**: las columnas numéricas usan coma decimal (`"128,00"`) y se leen
  como texto — deben convertirse.
- **Series cortas y desbalanceadas**: no todos los municipios/cultivos tienen datos en los 7 años
  ni en ambos semestres.
- **Heterogeneidad**: 165 cultivos distintos, con escalas de área y producción muy diferentes
  (ej. papa vs. flores), lo que genera fuerte asimetría (*skewness*) y valores atípicos legítimos.
- **Relación determinística conocida**: `Producción = Rendimiento × Área cosechada`, lo cual es a
  la vez una oportunidad (permite una vía de estimación indirecta) y un riesgo de fuga de
  información (*data leakage*) si no se maneja con cuidado en el modelado.


## 2. Justificación del Uso de Inteligencia Artificial

Aunque `Producción = Rendimiento × Área cosechada` es una identidad matemática, **ni el rendimiento
ni el área cosechada del futuro se conocen de antemano**: ambos dependen de clima, plagas, precios,
decisiones de siembra y dinámicas propias de cada cultivo y municipio. Esto convierte el problema
en uno genuino de **predicción de series de tiempo** donde la IA aporta valor porque:

- Puede **aprender patrones históricos no lineales** (estacionalidad semestral, tendencias por
  cultivo, efectos climáticos recurrentes) que un promedio simple no captura.
- Permite **comparar dos estrategias de predicción** — directa (predecir producción) vs. indirecta
  (predecir rendimiento y área, y multiplicarlos) — y cuantificar cuál acumula menos error,
  algo que no se puede resolver solo con la fórmula.
- Escala a **miles de combinaciones municipio–cultivo** simultáneamente, algo inviable de modelar
  manualmente una por una.
- Permite **medir y anticipar la degradación del error con el horizonte de predicción** (+1, +2,
  +3, +4 periodos), información clave para decidir hasta qué tan lejos en el futuro es confiable
  una predicción.


## 3. Variable Objetivo

La variable a predecir es **`Producción`** (toneladas), para un cultivo específico en un municipio
específico, a distintos horizontes futuros (siguiente periodo, dos periodos adelante, etc.).

Se comparará esta predicción **directa** de la producción contra una predicción **indirecta**,
obtenida al predecir por separado `Rendimiento` y `Área cosechada`, y luego calcular:

$$\widehat{Producción}_{indirecta} = \widehat{Rendimiento} \times \widehat{Área\ cosechada}$$

frente a

$$\widehat{Producción}_{directa} = f_{IA}(\text{histórico de Producción})$$

Ambas se contrastarán contra la `Producción` real para cada horizonte $h \in \{+1,+2,+3,+4\}$
periodos, permitiendo analizar cómo se degrada el error de cada enfoque a medida que el horizonte
de predicción crece. **`Rendimiento` no es el objetivo final del proyecto**, ya que su relación con
`Producción` y `Área cosechada` es conocida matemáticamente; se explora en el EDA únicamente como
insumo para la vía indirecta.


## 4. Preparación del Entorno

Ejecuta la siguiente celda. Si estás en **Google Colab** y aún no has subido el archivo, se te
pedirá que lo selecciones desde tu computador (descárgalo primero desde datos.gov.co o desde el
material del curso).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# Carga del dataset directamente desde el repositorio de GitHub del proyecto.
# Si por alguna razón la URL no responde, cae de vuelta al flujo local / upload manual.
GITHUB_RAW_URL = "https://raw.githubusercontent.com/Jd-jpg-exe/Produccion-Agricola-Proyecto-IA/main/EVA_2019_2025_Agricola.csv"
FILENAME = "EVA_2019_2025_Agricola.csv"

try:
    df_raw = pd.read_csv(GITHUB_RAW_URL)
    print(f"Cargado desde GitHub: {GITHUB_RAW_URL}")
except Exception as e:
    print(f"No se pudo cargar desde GitHub ({e}). Buscando el archivo localmente...")
    try:
        df_raw = pd.read_csv(FILENAME)
    except FileNotFoundError:
        try:
            from google.colab import files
            print(f"No se encontró '{FILENAME}' en el entorno. Selecciona el archivo CSV del dataset EVA:")
            uploaded = files.upload()
            FILENAME = list(uploaded.keys())[0]
            df_raw = pd.read_csv(FILENAME)
        except ImportError:
            raise FileNotFoundError(
                f"No se encontró '{FILENAME}' ni se pudo descargar de GitHub. Sube el CSV manualmente."
            )

print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
df_raw.head()


In [ ]:
df_raw.info()


## 5. Limpieza y Preparación de Datos

Las columnas numéricas (`Área sembrada`, `Área cosechada`, `Producción`, `Rendimiento`) llegan como
texto con coma decimal (formato colombiano). Se convierten a `float`, se estandarizan tipos
categóricos, y se deriva el semestre a partir de `Periodo`.


In [ ]:
df = df_raw.copy()

# --- 5.1 Conversión de columnas numéricas con coma decimal -> float ---
num_cols = ["Área sembrada", "Área cosechada", "Producción", "Rendimiento"]
for c in num_cols:
    df[c] = (
        df[c].astype(str)
             .str.strip()
             .str.replace(".", "", regex=False)   # separador de miles, si lo hubiera
             .str.replace(",", ".", regex=False)  # coma decimal -> punto
    )
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- 5.2 Variables categóricas ---
cat_cols = ["Departamento", "Municipio", "Grupo cultivo", "Subgrupo", "Cultivo",
            "Desagregación cultivo", "Ciclo del cultivo", "Estado físico del cultivo"]
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip()

# --- 5.3 Semestre a partir de 'Periodo' ('2020', '2020A', '2020B') ---
df["Semestre"] = df["Periodo"].astype(str).str.extract(r"([AB])$").fillna("Único")

# --- 5.4 Llave temporal ordenable (para futuras series de tiempo) ---
sem_rank = {"A": 0, "B": 1, "Único": 0}
df["orden_periodo"] = df["Año"] * 2 + df["Semestre"].map(sem_rank)

print("Nulos generados por conversión numérica:")
print(df[num_cols].isna().sum())
df[num_cols + ["Año", "Semestre"]].describe(include="all")


**Estrategia de limpieza aplicada:**

- Los valores no convertibles a número (si existieran) quedan como `NaN` explícito en vez de
  romper el pipeline; se cuantifican arriba antes de decidir qué hacer con ellos.
- No se eliminan filas por valores extremos de forma automática: en este dataset, áreas o
  producciones muy grandes suelen ser **reales** (departamentos con vocación agrícola fuerte en un
  cultivo, p. ej. arroz en Casanare), así que los "outliers" se estudian en la sección 7 antes de
  decidir si se filtran, se transforman (log) o se dejan intactos.
- Se valida duplicados exactos de fila, que si existen se eliminan por ser registros repetidos, no
  información nueva.


In [ ]:
dup = df.duplicated().sum()
print(f"Filas duplicadas exactas: {dup}")
if dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Filas tras eliminar duplicados: {len(df):,}")


In [ ]:
# Nulos remanentes tras la limpieza numérica: se descartan solo si Producción (variable objetivo)
# quedó indefinida, pues sin ella el registro no aporta a la tarea de predicción.
before = len(df)
df = df.dropna(subset=["Producción"]).reset_index(drop=True)
print(f"Filas eliminadas por Producción nula: {before - len(df)} de {before}")
print(f"Filas finales: {len(df):,}")


## 6. Descripción General del Dataset

El dataset tiene 18 columnas. Se priorizan para el análisis detallado las variables más relevantes
para la variable objetivo (`Producción`): las numéricas directamente relacionadas
(`Área sembrada`, `Área cosechada`, `Rendimiento`) y las categóricas con mayor poder explicativo
(`Cultivo`, `Grupo cultivo`, `Departamento`, `Ciclo del cultivo`, `Estado físico del cultivo`,
`Año`, `Semestre`).


In [ ]:
print(f"Dimensiones finales: {df.shape}")
print(f"\nRango temporal: {df['Año'].min()}–{df['Año'].max()}, semestres: {sorted(df['Semestre'].unique())}")
print(f"\nCultivos únicos: {df['Cultivo'].nunique()}")
print(f"Municipios únicos: {df['Municipio'].nunique()}")
print(f"Departamentos únicos: {df['Departamento'].nunique()}")
df.describe()


## 7. Calidad de Datos: Valores Nulos y Atípicos

### 7.1 Valores nulos


In [ ]:
nulos = df.isna().sum()
pct = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({"nulos": nulos, "% nulos": pct}).query("nulos > 0").sort_values("nulos", ascending=False)
resumen_nulos if len(resumen_nulos) else print("No quedan valores nulos tras la limpieza.")


### 7.2 Valores atípicos (outliers)

Se usa el criterio de **rango intercuartílico (IQR)** por ser robusto ante distribuciones
asimétricas, muy comunes aquí por la enorme diferencia de escala entre cultivos (ej. hortalizas de
pequeña área vs. arroz o palma en miles de hectáreas).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), ["Área sembrada", "Área cosechada", "Producción", "Rendimiento"]):
    sns.boxplot(x=df[col], ax=ax, color="#4C72B0")
    ax.set_title(f"Boxplot — {col}")
plt.tight_layout()
plt.show()


In [ ]:
def resumen_outliers_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    n_out = ((serie < lo) | (serie > hi)).sum()
    return {"Q1": q1, "Q3": q3, "IQR": iqr, "límite_inf": lo, "límite_sup": hi,
            "n_outliers": n_out, "% outliers": round(n_out / len(serie) * 100, 2)}

pd.DataFrame({c: resumen_outliers_iqr(df[c]) for c in
              ["Área sembrada", "Área cosechada", "Producción", "Rendimiento"]}).T


**Estrategia frente a los outliers detectados:**

Dado el alto porcentaje de "outliers" bajo IQR (esperable con 165 cultivos de escalas muy
distintas), **no se eliminan del dataset**: son observaciones reales y valiosas (p. ej. un
municipio líder nacional en un cultivo). En su lugar:

- Para visualización, se usa **escala logarítmica** en las variables de área/producción/rendimiento,
  que son estrictamente positivas.
- Para el modelado futuro, se recomienda trabajar por **grupo (cultivo)** en vez de con el dataset
  agregado completo, y evaluar transformaciones (`log1p`) o modelos robustos a escala.
- Se conservan como candidatos a revisión manual solo los casos con `Rendimiento` implausible
  (ver validación de la fórmula en la sección 9), que sí podrían ser errores de captura.


## 8. Análisis Univariado — Variables Categóricas

Siguiendo el estándar de reporte: para variables categóricas se usa **gráfico de barras** con
**conteo y porcentaje**.


In [ ]:
def reporte_categorica(serie, titulo, top_n=10):
    conteo = serie.value_counts().head(top_n)
    pct = (conteo / len(serie) * 100).round(1)

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(x=conteo.values, y=conteo.index, ax=ax, color="#4C72B0")
    for i, (v, p) in enumerate(zip(conteo.values, pct.values)):
        ax.text(v, i, f"  {v:,} ({p}%)", va="center", fontsize=9)
    ax.set_title(f"Frecuencia — {titulo} (top {top_n})")
    ax.set_xlabel("Conteo de registros")
    plt.tight_layout()
    plt.show()

    tabla = pd.DataFrame({"conteo": conteo, "%": pct})
    return tabla

reporte_categorica(df["Cultivo"], "Cultivo", top_n=15)


In [ ]:
reporte_categorica(df["Departamento"], "Departamento", top_n=15)


In [ ]:
reporte_categorica(df["Ciclo del cultivo"], "Ciclo del cultivo", top_n=5)


In [ ]:
reporte_categorica(df["Estado físico del cultivo"], "Estado físico del cultivo", top_n=10)


**Lectura del reporte (ejemplo de cómo comunicarlo en la presentación):**

- El cultivo más frecuente en los registros es *Maíz* (~13% de las filas), seguido de *Yuca* y
  *Frijol*, reflejando su siembra generalizada en casi todos los municipios y semestres.
- La mayoría de los registros corresponde a cultivos de ciclo **Transitorio** (~61%) frente a
  **Permanente** (~39%), es decir, cultivos que se resiembran cada semestre dominan el dataset.
- El estado físico más común es **"En fresco"** (~68%), coherente con que la mayoría de cultivos
  reportados son hortalizas, frutas y tubérculos que no requieren transformación previa al reporte.


## 9. Análisis Univariado — Variables Numéricas

Para decidir si cada variable numérica se reporta como **simétrica** (media ± desv. estándar) o
**asimétrica** (mediana y rango/IQR), primero se grafica su histograma y se calcula el coeficiente
de asimetría (*skewness*): valores cercanos a 0 indican simetría; valores grandes (positivos o
negativos) indican sesgo.


In [ ]:
def reporte_numerica(serie, titulo, log_scale=True, skew_threshold=1.0):
    s = serie.dropna()
    skew = stats.skew(s)
    es_simetrica = abs(skew) < skew_threshold

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.histplot(s, kde=True, ax=axes[0], color="#4C72B0", bins=40)
    axes[0].set_title(f"Histograma — {titulo} (escala original)")
    axes[0].set_xlabel(titulo)

    s_log = np.log1p(s[s >= 0])
    sns.histplot(s_log, kde=True, ax=axes[1], color="#55A868", bins=40)
    axes[1].set_title(f"Histograma — {titulo} (log1p, para visualizar mejor)")
    axes[1].set_xlabel(f"log1p({titulo})")
    plt.tight_layout()
    plt.show()

    print(f"Asimetría (skewness) = {skew:.2f} -> {'SIMÉTRICA' if es_simetrica else 'ASIMÉTRICA'} "
          f"(umbral |skew| < {skew_threshold})")

    if es_simetrica:
        media, desv = s.mean(), s.std()
        print(f"Reporte sugerido: '{titulo} promedio es {media:,.2f} ± {desv:,.2f}'.")
        return {"tipo": "simétrica", "media": media, "desv_std": desv}
    else:
        mediana = s.median()
        q1, q3 = s.quantile([0.25, 0.75])
        print(f"Reporte sugerido: '{titulo} tiene una mediana de {mediana:,.2f} "
              f"y un rango intercuartílico de ({q1:,.2f} - {q3:,.2f})'.")
        return {"tipo": "asimétrica", "mediana": mediana, "min": s.min(), "max": s.max(),
                "Q1": q1, "Q3": q3}

resultados_num = {}
for col in ["Área sembrada", "Área cosechada", "Producción", "Rendimiento"]:
    print(f"\n{'='*70}\n{col}\n{'='*70}")
    resultados_num[col] = reporte_numerica(df[col], col)


In [ ]:
pd.DataFrame(resultados_num).T


**Lectura:** las cuatro variables numéricas resultan **fuertemente asimétricas** (skewness alto),
lo cual es esperable: coexisten cultivos de traspatio (fracciones de hectárea) con agroindustrias
de miles de hectáreas. Por eso se reportan con **mediana y rango intercuartílico**, y se recomienda
usar transformación logarítmica para cualquier modelo que asuma distribuciones más simétricas.


## 10. Variable Objetivo: `Producción` en Detalle


In [ ]:
top_grupos = df.groupby("Grupo cultivo")["Producción"].sum().sort_values(ascending=False).head(8).index

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df[df["Grupo cultivo"].isin(top_grupos)],
            x="Grupo cultivo", y="Producción", ax=ax, color="#C44E52")
ax.set_yscale("log")
ax.set_title("Distribución de Producción (escala log) por Grupo de cultivo (top 8 por producción total)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
produccion_anual = df.groupby("Año")["Producción"].sum() / 1000  # miles de toneladas

fig, ax = plt.subplots(figsize=(9, 4.5))
produccion_anual.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Producción total (miles de toneladas)")
ax.set_title("Producción agrícola total reportada por año (todos los cultivos)")
plt.tight_layout()
plt.show()

produccion_anual


*Nota:* la caída visible en los años más recientes puede deberse a que el reporte de esos
periodos aún no está completo (rezago administrativo de la EVA), no necesariamente a una caída
real de producción. Esto es relevante para el modelado: los últimos periodos podrían no ser
comparables directamente con los históricos completos.


## 11. Correlaciones y Validación de la Relación Producción = Rendimiento × Área Cosechada


In [ ]:
corr_cols = ["Área sembrada", "Área cosechada", "Rendimiento", "Producción"]
corr = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación de Spearman entre variables numéricas")
plt.tight_layout()
plt.show()


In [ ]:
# Validación de la identidad Producción = Rendimiento x Área cosechada
df["Produccion_calculada"] = df["Rendimiento"] * df["Área cosechada"]
df["error_relativo_formula"] = (df["Producción"] - df["Produccion_calculada"]).abs() / df["Producción"].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(6, 6))
sample = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
ax.scatter(sample["Produccion_calculada"], sample["Producción"], alpha=0.3, s=10)
lims = [0, sample[["Produccion_calculada", "Producción"]].quantile(0.99).max()]
ax.plot(lims, lims, "r--", label="y = x (ajuste perfecto)")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Producción calculada (Rendimiento × Área cosechada)")
ax.set_ylabel("Producción reportada")
ax.set_title("Validación de la identidad Producción = Rendimiento × Área cosechada")
ax.legend()
plt.tight_layout()
plt.show()

print("Error relativo de la fórmula (debería ser ~0 salvo redondeos de reporte):")
print(df["error_relativo_formula"].describe())


La identidad se cumple casi exactamente (el pequeño error remanente proviene del redondeo de
`Rendimiento` a 2 decimales en el reporte oficial). Esto confirma que la vía **indirecta**
(predecir `Rendimiento` y `Área cosechada` por separado y multiplicarlos) es matemáticamente válida
como punto de comparación contra la predicción **directa** de `Producción`.


## 12. Comportamiento Temporal y Preparación para el Análisis por Horizonte

El objetivo final del proyecto compara el error de predicción a distintos horizontes
(+1, +2, +3, +4 periodos). Como base para esa fase, se construye aquí la serie temporal por
combinación **Municipio–Cultivo** y se inspecciona su comportamiento para un caso representativo.


In [ ]:
serie = (df.groupby(["Municipio", "Cultivo", "Año", "Semestre", "orden_periodo"])
           .agg(Producción=("Producción", "sum"),
                Área_cosechada=("Área cosechada", "sum"),
                Rendimiento=("Rendimiento", "mean"))
           .reset_index()
           .sort_values("orden_periodo"))

conteo_periodos = serie.groupby(["Municipio", "Cultivo"])["orden_periodo"].nunique()
print(f"Combinaciones Municipio-Cultivo con al menos 8 periodos históricos: "
      f"{(conteo_periodos >= 8).sum():,} de {len(conteo_periodos):,}")

# Caso ilustrativo: el cultivo más reportado, en su municipio con más periodos disponibles
cultivo_top = df["Cultivo"].value_counts().idxmax()
candidatos = conteo_periodos.loc[:, cultivo_top].sort_values(ascending=False)
municipio_top = candidatos.index[0]

ejemplo = serie[(serie["Municipio"] == municipio_top) & (serie["Cultivo"] == cultivo_top)]
print(f"\nEjemplo ilustrativo: Cultivo = {cultivo_top}, Municipio = {municipio_top}, "
      f"periodos disponibles = {len(ejemplo)}")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(ejemplo["orden_periodo"], ejemplo["Producción"], marker="o")
ax.set_xlabel("Periodo (orden temporal)")
ax.set_ylabel("Producción (t)")
ax.set_title(f"Serie histórica de Producción — {cultivo_top} en {municipio_top}")
plt.tight_layout()
plt.show()


Este tipo de serie —una por cada combinación Municipio–Cultivo— es la unidad base sobre la que,
en la siguiente fase del proyecto, se construirán variables rezagadas (*lags*) para entrenar y
comparar los modelos de predicción **directa** de `Producción` frente a la vía **indirecta**
(`Rendimiento` × `Área cosechada`), evaluando el error (p. ej. MAE o RMSE) en cada horizonte
+1, +2, +3 y +4 periodos, tal como se plantea en la tabla comparativa del proyecto.


## 13. Calidad de Datos y Patrones — Resumen

- **Nulos:** tras convertir las columnas numéricas de texto (coma decimal) a `float`, los nulos
  remanentes son mínimos; las filas sin `Producción` (variable objetivo) se descartaron por no
  aportar a la tarea de predicción.
- **Outliers:** abundantes bajo el criterio IQR, pero en su mayoría **legítimos** dada la enorme
  heterogeneidad entre 165 cultivos distintos. Se decidió **no eliminarlos**, usar **escala
  logarítmica** para visualizarlos y trabajar el modelado **por cultivo** para evitar mezclar
  escalas incompatibles.
- **Duplicados:** se verificaron y removieron de existir filas exactamente repetidas.
- **Consistencia interna:** se validó que `Producción ≈ Rendimiento × Área cosechada` se cumple
  con error prácticamente nulo, confirmando la calidad del dato y habilitando la comparación
  directa vs. indirecta planteada en el proyecto.
- **Patrones interesantes:**
  - Maíz, Yuca y Frijol concentran la mayor cantidad de registros (cultivos casi universales en el
    territorio nacional).
  - Los cultivos Transitorios dominan en número de registros, pero los Permanentes suelen aportar
    más producción total por hectárea en cultivos específicos (ej. palma, caña).
  - Se observa posible sub-reporte en los periodos más recientes (2024–2025), probablemente por
    rezago administrativo, lo que debe tenerse en cuenta al definir la ventana de entrenamiento.


## 14. Conclusiones y Próximos Pasos

**Hallazgos principales:**
1. El dataset EVA es rico y consistente, pero requiere limpieza de formato numérico y manejo
   cuidadoso de la fuerte asimetría entre cultivos de escalas muy distintas.
2. La identidad `Producción = Rendimiento × Área cosechada` se confirma empíricamente, validando el
   diseño de comparación directa vs. indirecta propuesto para el proyecto.
3. Existen suficientes combinaciones Municipio–Cultivo con historia larga (≥8 periodos) para
   construir series de tiempo y evaluar el error por horizonte de predicción.

**Próximos pasos (preprocesamiento y modelado):**
- Seleccionar un subconjunto de combinaciones Municipio–Cultivo con historia suficiente y sin
  demasiados huecos temporales.
- Construir variables rezagadas (*lags*) de `Producción`, `Rendimiento` y `Área cosechada`, además
  de variables estacionales (semestre) y de tendencia.
- Entrenar modelos de predicción **directa** de `Producción` y modelos separados para
  `Rendimiento` y `Área cosechada` (vía **indirecta**), a horizontes +1 a +4 periodos.
- Comparar el error (MAE/RMSE/MAPE) de ambas vías por horizonte y construir la tabla comparativa
  planteada en el objetivo del proyecto.
- Evaluar transformación logarítmica y/o modelado por grupo de cultivo para mitigar el efecto de
  la asimetría y los valores atípicos identificados en este EDA.
